# 表情识别 · 离线四模型软投票

将权重 ZIP 作为 Dataset 加入，再加入比赛数据，启用 GPU 后 **Run All**。输出 `submission.csv`，无需联网或重新训练。

默认是项目已验证的 V8 融合；它不代表 Kaggle 最优。V10 是不同开发划分的独立候选，可在下格切换。

In [ ]:
from pathlib import Path
import os

INPUT_ROOT = Path(os.environ.get("EXPRESSION_INPUT_ROOT", "/kaggle/input"))
WORKING = Path(os.environ.get("EXPRESSION_WORKING", "/kaggle/working"))
BUNDLE_PATH = os.environ.get("EXPRESSION_BUNDLE_PATH") or None  # ZIP 或 expression_bundle 目录；通常留空
TEST_DIR = os.environ.get("EXPRESSION_TEST_DIR") or None  # test 图片所在目录；通常留空
MODE = os.environ.get("EXPRESSION_MODE", "ensemble_v8")  # 或 "hse_v10"
DEVICE = "cuda"
BATCH_SIZE = 16
BLUR_PROBE_IMAGES = 64  # 无标签合成模糊稳定性检查；设0可跳过
WORKING.mkdir(parents=True, exist_ok=True)


## 加载离线代码与权重

支持 Kaggle 自动解压后的目录，或仍保留的 ZIP。逐文件 SHA256 校验后加载真实模型。

In [ ]:
import importlib.util
import sys
import zipfile

if BUNDLE_PATH and Path(BUNDLE_PATH).is_dir():
    runtime_file = Path(BUNDLE_PATH) / "inference.py"
else:
    runtime_files = list(INPUT_ROOT.rglob("expression_bundle/inference.py")) if not BUNDLE_PATH else []
    if len(runtime_files) == 1:
        runtime_file = runtime_files[0]
    else:
        archives = [Path(BUNDLE_PATH)] if BUNDLE_PATH else list(INPUT_ROOT.rglob("expression_pretrained_weights.zip"))
        if len(archives) != 1:
            raise FileNotFoundError("Add Input: expression_pretrained_weights.zip（或填写 BUNDLE_PATH）")
        runtime_file = WORKING / "expression_inference.py"
        with zipfile.ZipFile(archives[0]) as z:
            runtime_file.write_bytes(z.read("expression_bundle/inference.py"))
spec = importlib.util.spec_from_file_location("expression_runtime", runtime_file)
runtime = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = runtime
spec.loader.exec_module(runtime)
BUNDLE = runtime.locate_bundle(INPUT_ROOT, WORKING, BUNDLE_PATH)
runtime.configure(BUNDLE, DEVICE)
manifest = runtime.read_json(BUNDLE / "manifest.json")
print("Mode:", MODE, "Classes:", runtime.CLASSES)


## 定位比赛图片

严格采用比赛模板的7065个 ID 与顺序。只读取测试图；不需要训练集或标签。

In [ ]:
IDS = runtime.template_ids(BUNDLE / "submission_template.csv")
PATHS = runtime.locate_images(INPUT_ROOT, WORKING, IDS, TEST_DIR)
import hashlib
image_digest = hashlib.sha256()
for p in PATHS:
    image_digest.update(runtime.sha256(p).encode("ascii"))
if image_digest.hexdigest() != manifest["competition_image_content_sha256"]:
    raise ValueError("测试图内容与此比赛的冻结输入不同，请检查是否加入正确的比赛数据集")
print("Competition images verified:", len(PATHS))


## 串行推断与软投票

每路采用原图+水平翻转。前64张额外做合成模糊测试，仅报告稳定性，不改最终投票。可在日志中查看每路进度。

In [ ]:
PROBABILITIES, VOTES, RUN_INFO = runtime.predict(
    BUNDLE, PATHS, WORKING, MODE, DEVICE, BATCH_SIZE, BLUR_PROBE_IMAGES)
receipt = runtime.write_outputs(BUNDLE, IDS, PATHS, PROBABILITIES, VOTES, WORKING, RUN_INFO)
print(receipt)


## 模糊图和低置信度检查

清晰度为启发式指标，softmax 概率未校准。真实模糊图准确率需要独立标注确认；合成模糊的预测一致性不能替代准确率。诊断表不改变 `ID,Emotion` 提交格式。

In [ ]:
import csv
import html
from IPython.display import display, FileLink, HTML
with (WORKING / "prediction_diagnostics.csv").open(encoding="utf-8", newline="") as f:
    diagnostics = list(csv.DictReader(f))
diagnostics.sort(key=lambda row: (row["uncertain_heuristic"] != "True", float(row["margin"])))
columns = ["ID", "class_name", "top1_probability", "top2_class", "margin", "low_sharpness_heuristic"]
table = "<table><tr>" + "".join("<th>" + html.escape(c) + "</th>" for c in columns) + "</tr>"
for row in diagnostics[:20]:
    table += "<tr>" + "".join("<td>" + html.escape(row[c]) + "</td>" for c in columns) + "</tr>"
display(HTML(table + "</table>"))
display(FileLink(str(WORKING / "submission.csv")))
display(FileLink(str(WORKING / "prediction_diagnostics.csv")))
if (WORKING / "blur_stability.json").exists():
    print(runtime.read_json(WORKING / "blur_stability.json"))


权重来源：[DINOv2](https://github.com/facebookresearch/dinov2)、[HSE/EmotiEffLib](https://github.com/sb-ai-lab/EmotiEffLib)、[DDAMFN++](https://github.com/SainingZhang/DDAMFN)。默认融合旧开发集72.4398%；V10另一划分67.7541%，不作跨划分排名。开发集被反复使用、预训练重叠未知，不承诺 Kaggle 提分。详细来源和许可说明见 ZIP 内 README。